In [69]:
%pip install einx

Note: you may need to restart the kernel to use updated packages.


In [97]:
%%writefile run_multihead_self_attention.py

import torch
import torch.nn as nn
from einops import rearrange, einsum
import einx
from cs336_basics.run_linear import Linear
from cs336_basics.run_scaled_dot_production_attention import scaled_dot_product_attention
from cs336_basics.run_rope import RoPE


class CausalMultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, device=None, dtype=None, rope=None):
        super(CausalMultiHeadSelfAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = self.d_q = self.d_v = int(d_model / num_heads)
        self.linear_q = Linear(d_model, num_heads * self.d_q)
        self.linear_k = Linear(d_model, num_heads * self.d_k)
        self.linear_v = Linear(d_model, num_heads * self.d_v)
        self.linear_o = Linear(num_heads * self.d_v, d_model)
        self.rope = rope
        
    
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor=None) -> torch.Tensor:
        # x.shape = (batch, seq, d_model)
        batch_size, seq_len, _ = x.shape
        # token_positions = torch.arange(seq_len)
        # Q.shape = (batch, num_heads, seq, d_q)
        Q = rearrange(self.linear_q(x), "b s (h d) -> b h s d", h=self.num_heads)
        # K.shape = (batch, num_heads, seq, d_k)
        K = rearrange(self.linear_k(x), "b s (h d) -> b h s d", h=self.num_heads)
        # V.shape = (batch, num_heads, seq, d_v)
        V = rearrange(self.linear_v(x), "b s (h d) -> b h s d", h=self.num_heads)
        
        if token_positions is not None and self.rope:
            Q = self.rope(Q, token_positions)
            K = self.rope(K, token_positions)
                
        # mask.shape = (seq_q, seq_k)
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool))
        
        attention = scaled_dot_product_attention(Q, K, V, mask=mask)
        attention = rearrange(attention, "b h s d -> b s (h d)")
        
        return self.linear_o(attention)

Overwriting run_multihead_self_attention.py


In [98]:
%reload_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import torch.nn as nn


from cs336_basics.run_multihead_self_attention import CausalMultiHeadSelfAttention


test_model = CausalMultiHeadSelfAttention(64, 8)
x = torch.rand(100,  100, 64)
test_model(x)

tensor([[[-1.5839e-02,  3.3166e-01, -1.5843e-01,  ..., -2.1397e-01,
          -2.4399e-01,  1.4134e-01],
         [ 1.2121e-02,  1.7928e-01, -1.0258e-01,  ..., -1.9419e-01,
          -2.1589e-01,  6.8155e-02],
         [ 1.1600e-01,  1.7245e-01,  1.1997e-01,  ..., -2.8914e-01,
          -1.8793e-02, -1.2696e-01],
         ...,
         [ 3.4201e-04,  2.3805e-01, -1.4438e-01,  ..., -2.6734e-01,
          -3.1201e-01, -1.9570e-02],
         [-3.8652e-03,  2.3504e-01, -1.4352e-01,  ..., -2.5913e-01,
          -3.1299e-01, -2.2412e-02],
         [-7.7376e-03,  2.3756e-01, -1.4884e-01,  ..., -2.5487e-01,
          -3.1906e-01, -1.2455e-02]],

        [[ 1.0564e-01,  4.6742e-01, -3.4032e-01,  ..., -3.5706e-01,
          -4.3661e-01,  6.9730e-02],
         [-3.1434e-02,  1.8249e-01, -1.9094e-02,  ..., -2.8527e-01,
          -6.4676e-01,  4.8553e-02],
         [ 6.3118e-02,  2.2805e-01, -1.6553e-02,  ..., -2.5085e-01,
          -5.4786e-01,  2.3786e-01],
         ...,
         [-2.9267e-02,  1